In [ ]:
%pip install pandas
%pip install numpy
import pandas as pd
import numpy as np
import os

def procesar_ciclo_limpio(letra_ciclo, ruta_carpeta):

    print(f"Procesando ciclo {letra_ciclo}...") 
    
    # 1. LECTURA DIRECTA
    df_demo = pd.read_sas(os.path.join(ruta_carpeta, f"DEMO_{letra_ciclo}.XPT"))[['SEQN', 'RIDAGEYR', 'RIAGENDR']]
    df_biopro = pd.read_sas(os.path.join(ruta_carpeta, f"BIOPRO_{letra_ciclo}.XPT"))[['SEQN', 'LBXSAL', 'LBXSATSI', 'LBXSASSI', 'LBXSGTSI', 'LBXSAPSI', 'LBXSBU', 'LBXSCA', 'LBXSCH', 'LBXSIR', 'LBXSTB', 'LBXSTP', 'LBXSTR', 'LBXSUA', 'LBXSGL']]
    df_cbc = pd.read_sas(os.path.join(ruta_carpeta, f"CBC_{letra_ciclo}.XPT"))[['SEQN', 'LBXWBCSI', 'LBDNENO', 'LBDLYMNO', 'LBDMONO', 'LBXHGB', 'LBXRDW', 'LBXPLTSI']]
    df_alb = pd.read_sas(os.path.join(ruta_carpeta, f"ALB_CR_{letra_ciclo}.XPT"))[['SEQN', 'URXUMA', 'URXUCR']]
    df_bmx = pd.read_sas(os.path.join(ruta_carpeta, f"BMX_{letra_ciclo}.XPT"))[['SEQN', 'BMXBMI', 'BMXWAIST']]
    
    # Ajuste dinámico para Cotinina (Ciclos E, F, G usan COTNAL)
    if letra_ciclo in ['E', 'F', 'G']:
        df_cot = pd.read_sas(os.path.join(ruta_carpeta, f"COTNAL_{letra_ciclo}.XPT"))[['SEQN', 'LBXCOT']]
    else:
        df_cot = pd.read_sas(os.path.join(ruta_carpeta, f"COT_{letra_ciclo}.XPT"))[['SEQN', 'LBXCOT']]
        
    df_ghb = pd.read_sas(os.path.join(ruta_carpeta, f"GHB_{letra_ciclo}.XPT"))[['SEQN', 'LBXGH']]
    df_hepb = pd.read_sas(os.path.join(ruta_carpeta, f"HEPBD_{letra_ciclo}.XPT"))[['SEQN', 'LBDHBG', 'LBDHD', 'LBXHBC']]
    df_hepc = pd.read_sas(os.path.join(ruta_carpeta, f"HEPC_{letra_ciclo}.XPT"))[['SEQN', 'LBXHCR']]
    
    # Vitamina D unificada bajo el nombre LBXVIDMS para todos los ciclos
    df_vitd = pd.read_sas(os.path.join(ruta_carpeta, f"VID_{letra_ciclo}.XPT"))[['SEQN', 'LBXVIDMS']]
    
    df_alq = pd.read_sas(os.path.join(ruta_carpeta, f"ALQ_{letra_ciclo}.XPT"))[['SEQN', 'ALQ130']]
    df_bpq = pd.read_sas(os.path.join(ruta_carpeta, f"BPQ_{letra_ciclo}.XPT"))[['SEQN', 'BPQ020']]
    df_diq = pd.read_sas(os.path.join(ruta_carpeta, f"DIQ_{letra_ciclo}.XPT"))[['SEQN', 'DIQ010']]
    df_smq = pd.read_sas(os.path.join(ruta_carpeta, f"SMQ_{letra_ciclo}.XPT"))[['SEQN', 'SMQ020']]

    # 2. CONCATENACIÓN HORIZONTAL SECUENCIAL
    df_master = (df_demo.merge(df_biopro, on='SEQN', how='inner')
                        .merge(df_cbc, on='SEQN', how='inner')
                        .merge(df_alb, on='SEQN', how='inner')
                        .merge(df_bmx, on='SEQN', how='inner')
                        .merge(df_cot, on='SEQN', how='inner')
                        .merge(df_ghb, on='SEQN', how='inner')
                        .merge(df_hepb, on='SEQN', how='inner')
                        .merge(df_hepc, on='SEQN', how='inner')
                        .merge(df_vitd, on='SEQN', how='inner')
                        .merge(df_alq, on='SEQN', how='inner')
                        .merge(df_bpq, on='SEQN', how='inner')
                        .merge(df_diq, on='SEQN', how='inner')
                        .merge(df_smq, on='SEQN', how='inner'))

    # 3. CÁLCULO DE ÍNDICES CLÍNICOS
    df_master['ICA'] = (df_master['URXUMA'] / df_master['URXUCR']) * 100
    df_master['RIA'] = df_master['LBDNENO'] / df_master['LBDLYMNO']
    df_master['RAG'] = df_master['LBXSAL'] / (df_master['LBXSTP'] - df_master['LBXSAL'])
    df_master.replace([np.inf, -np.inf], np.nan, inplace=True)

    # 4. LIMPIEZA FINAL DE REDUNDANCIAS
    df_master.drop(columns=['URXUMA', 'URXUCR', 'LBDNENO', 'LBDLYMNO', 'LBXSTP'], inplace=True)
    df_master['CICLO_ORIGEN'] = letra_ciclo
    
    return df_master

# ------------------------------------------
# EJECUCIÓN
# ------------------------------------------
lista_dataframes = []

# Rutas a carpetas locales
diccionario_rutas = {
    'E': 'datos_nhanes/2007_2008_E/',
    'F': 'datos_nhanes/2009_2010_F/',
    'G': 'datos_nhanes/2011_2012_G/',
    'H': 'datos_nhanes/2013_2014_H/',
    'I': 'datos_nhanes/2015_2016_I/',
    'J': 'datos_nhanes/2017_2018_J/'
}

for letra, ruta in diccionario_rutas.items():
    try:
        df_anual = procesar_ciclo_limpio(letra, ruta)
        lista_dataframes.append(df_anual)
        print(f"  -> Ciclo {letra} acoplado. Filas: {df_anual.shape[0]}")
    except Exception as e:
        print(f"  -> ERROR crítico en el ciclo {letra}: {e}")

# 5. CONSOLIDACIÓN FINAL
df_laboratorios_completo = pd.concat(lista_dataframes, ignore_index=True)

print("\n" + "-"*50)
print("¡Dataframe listo!")
print(f"Pacientes totales: {df_laboratorios_completo.shape[0]}")
print(f"Predictores biológicos: {df_laboratorios_completo.shape[1]}")
print("-"*50)

Procesando ciclo E...
  -> Ciclo E acoplado. Filas: 5707
Procesando ciclo F...
  -> Ciclo F acoplado. Filas: 6059
Procesando ciclo G...
  -> Ciclo G acoplado. Filas: 5615
Procesando ciclo H...
  -> Ciclo H acoplado. Filas: 5924
Procesando ciclo I...
  -> Ciclo I acoplado. Filas: 5735
Procesando ciclo J...
  -> Ciclo J acoplado. Filas: 5533

--------------------------------------------------
¡Dataframe listo!
Pacientes totales: 34573
Predictores biológicos: 38
--------------------------------------------------


In [7]:
import glob

print("Extrayendo archivos de supervivencia del NCHS...")

# 1. Definir coordenadas del diccionario de datos (ancho fijo)
coordenadas_columnas = [(0, 6), (14, 15), (15, 16), (16, 19), (19, 22)]
nombres_columnas = ['SEQN', 'ELIGSTAT', 'MORTSTAT', 'UCOD_LEADING', 'PERMTH_INT']

# 2. Leer dinámicamente todos los archivos .dat en la carpeta local
rutas_mortalidad = glob.glob("datos_mortalidad/*.dat")
lista_mortalidad = []

for ruta in rutas_mortalidad:
    df_mort_temp = pd.read_fwf(ruta, colspecs=coordenadas_columnas, names=nombres_columnas)
    lista_mortalidad.append(df_mort_temp)

# 3. Consolidar el archivo maestro de mortalidad
df_mortalidad_total = pd.concat(lista_mortalidad, ignore_index=True)

# Asegurar que la llave primaria sea de tipo entero en ambos DataFrames para un cruce perfecto
df_mortalidad_total['SEQN'] = df_mortalidad_total['SEQN'].astype(int)
df_laboratorios_completo['SEQN'] = df_laboratorios_completo['SEQN'].astype(int)

# 4. Filtrar solo pacientes con seguimiento válido (ELIGSTAT == 1)
df_mortalidad_total = df_mortalidad_total[df_mortalidad_total['ELIGSTAT'] == 1].drop(columns=['ELIGSTAT'])

# 5. Merge (Características + Supervivencia)
df_maestro = df_laboratorios_completo.merge(df_mortalidad_total, on='SEQN', how='inner')

print("\n" + "-"*50)
print("¡Dataset Meastro listo!")
print(f"Pacientes listos para imputación: {df_maestro.shape[0]}")
print("-"*50)

Extrayendo archivos de supervivencia del NCHS...

--------------------------------------------------
¡Dataset Meastro listo!
Pacientes listos para imputación: 34480
--------------------------------------------------


In [8]:
import pandas as pd

# 1. FILTRO DEMOGRÁFICO INICIAL (Solo adultos)
df_adultos = df_maestro[df_maestro['RIDAGEYR'] >= 18].copy()

# ------------------------------------------
# NIVEL 0: Tolerancia Cero (Target y Demografía Básica)
# ------------------------------------------
# CORRECCIÓN! Se quitó UCOD_LEADING del dropna, ya que solo dejaba a pacientes muertos 
columnas_intocables = ['PERMTH_INT', 'MORTSTAT', 'RIDAGEYR', 'RIAGENDR']
df_filtrado = df_adultos.dropna(subset=columnas_intocables).copy()

# 1. Alcohol: Los que saltaron la pregunta de "tragos diarios" son no bebedores (0)
df_filtrado['ALQ130'] = df_filtrado['ALQ130'].fillna(0)

# 2. Hepatitis C (LBXHCR): Si no se midió ARN, asumimos clínicamente carga viral negativa (0)
# (Ajusta el '0' si NHANES usa un código numérico específico para 'Negativo')
df_filtrado['LBXHCR'] = df_filtrado['LBXHCR'].fillna(0)

# ------------------------------------------
# PREPARACIÓN PARA NIVELES 1 Y 2
# ------------------------------------------
# Calcilo de los porcentajes de vacíos
porcentaje_nulos_fila = df_filtrado.isnull().mean(axis=1)

# Núcleo  
variables_clave_biologicas = ['BMXBMI', 'LBXSGL', 'LBXSCH', 'LBXGH']

# ------------------------------------------
# APLICACIÓN DE REGLAS NIVEL 1 y 2
# ------------------------------------------
excluir_nivel_1 = porcentaje_nulos_fila > 0.30

falta_biologia = df_filtrado[variables_clave_biologicas].isnull().any(axis=1)
excluir_nivel_2 = (porcentaje_nulos_fila >= 0.20) & falta_biologia

df_para_imputacion = df_filtrado[~(excluir_nivel_1 | excluir_nivel_2)].copy()

# ------------------------------------------
# REPORTE DE EXITO
# ------------------------------------------
pacientes_retenidos = df_para_imputacion.shape[0]
pacientes_eliminados = df_adultos.shape[0] - pacientes_retenidos

print("-" * 50)
print("FILTRO DE DATOS FALTANTES - VERSIÓN FINAL")
print(f"Pacientes eliminados por mala calidad de datos: {pacientes_eliminados}")
print(f"Pacientes retenidos para MICE: {pacientes_retenidos}")
print("-" * 50)

--------------------------------------------------
FILTRO DE DATOS FALTANTES - VERSIÓN FINAL
Pacientes eliminados por mala calidad de datos: 2238
Pacientes retenidos para MICE: 32242
--------------------------------------------------


In [12]:
%pip install scikit-learn
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
import pandas as pd
import numpy as np

print("Preparando la matriz")

# ------------------------------------------
# 1. SEPARAR EL TARGET DE LOS PREDICTORES
# ------------------------------------------

columnas_protegidas = ['SEQN', 'PERMTH_INT', 'MORTSTAT', 'UCOD_LEADING', 'CICLO_ORIGEN']
columnas_predictores = [col for col in df_para_imputacion.columns if col not in columnas_protegidas]

X_incompleto = df_para_imputacion[columnas_predictores]
# ------------------------------------------
# 2. CONFIGURACIÓN DEL ALGORITMO DE IMPUTACIÓN
# ------------------------------------------

imputador = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,        
    tol=1e-3,
    initial_strategy='median', 
    random_state=42,   
    verbose=2           
)
# ------------------------------------------
# 3. EJECUCIÓN 
# ------------------------------------------

print("Iniciando imputación de datos faltantes. Por favor, espera...")
X_imputado = imputador.fit_transform(X_incompleto)

# ------------------------------------------
# 4. RECONSTRUCCIÓN DEL DATAFRAME MAESTRO
# ------------------------------------------

# Array numpy de vuelta a Pandas 
df_predictores_limpios = pd.DataFrame(
    X_imputado, 
    columns=columnas_predictores, 
    index=df_para_imputacion.index
)

# Re-incorporación de columnas protegidas
df_tesis_imputado = pd.concat([df_para_imputacion[columnas_protegidas], df_predictores_limpios], axis=1)

print("\n" + "-"*50)
print("¡IMPUTACIÓN COMPLETADA!")
print(f"Valores nulos restantes en el dataset: {df_tesis_imputado.isnull().sum().sum()}")
print("-"*50)


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
Preparando la matriz
Iniciando imputación de datos faltantes. Por favor, espera...
[IterativeImputer] Completing matrix with shape (32242, 36)
[IterativeImputer] Ending imputation round 1/10, elapsed time 1.46
[IterativeImputer] Change: 1188.5086322373281, scaled tolerance: 21.15217391304348 
[IterativeImputer] Ending imputation round 2/10, elapsed time 2.92
[IterativeImputer] Change: 56.207525898051685, scaled tolerance: 21.15217391304348 
[IterativeImputer] Ending imputation round 3/10, elapsed time 4.41
[IterativeImputer] Change: 24.141640629760573, scaled tolerance: 21.15217391304348 
[IterativeImputer] Ending imputation round 4/10, elapsed time 5.85
[IterativeImputer] Change: 14.815052982065126, scaled tolerance: 21.15217391304348 
[IterativeImputer] Early stopping criterion reached.

--------------------------------------------------
¡IMPUTACIÓN COMPLETADA!
Valores nulos restantes en el dataset: 29305
-------------

In [13]:
# Verificación de nulos
nulos_restantes = df_tesis_imputado.isnull().sum()
print("Variables que aún tienen valores nulos:")
print(nulos_restantes[nulos_restantes > 0])

# Exportar el DataFrame Maestro a un archivo CSV
nombre_archivo = "df_maestro_imputado_final.csv"
df_tesis_imputado.to_csv(nombre_archivo, index=False)

print(f"\nDataset guardado  como '{nombre_archivo}'")
print("Dataset listo para usarse en el entrenamiento.")

Variables que aún tienen valores nulos:
UCOD_LEADING    29305
dtype: int64

Dataset guardado  como 'df_maestro_imputado_final.csv'
Dataset listo para usarse en el entrenamiento.


Los **29305** datos faltantes corresponden a la columna **"UCOD_LEADING"**, que es la variable que muestra la causa de muerte de los pacientes, que logicamente esta vacía si el paciente esta vivo, por lo que se dejó como columna protegida y reincorporada (Previamente al poner esta variable en los filtros de datos faltantes dejaba solo a 2000 pacientes, ya que aproximadamente el 90% de los pacientes están vivos en el último año del dataset         )